# 1.2 — Load a Policy Trained by SB3 or mjlab

This notebook drives the
env `ENV_ID` (default `myoElbowPose1D6MRandom-v0`, set in the first code cell) with a checkpoint from a default training run of that environment,
and renders the rollout to MP4.

**What you'll learn:**
- How to load an mjlab (RSL-RL) checkpoint and a Stable-Baselines3 checkpoint behind one `act(obs)` function
- How to run the rollout loop: `reset → step → render`
- How to generate and store a video of the rollout

**Prerequisites:** Completed notebook 1.1. Nothing else is needed for the defaults: without a local run the notebook uses the repository's default policy for `ENV_ID` from `baselines/checkpoints/<ENV_ID>/` (32 envs; see its README for success rates and unconverged snapshots). To use your own, provide a checkpoint from one of:
- mjlab: `python scripts/train_mjlab.py <task-id>` (stores `logs/rsl_rl/myo_elbow_pose/<run>/model_<iter>.pt`), needs `pip install -e ".[mjlab]"`
- SB3: notebook 2.1 (stores `files/2.1/ElbowPose_policy.zip`), needs `pip install stable-baselines3`

Search order: your `logs/rsl_rl/<experiment>/<run>/model_*.pt` (newest), then `baselines/checkpoints/<ENV_ID>/`, then the SB3 zip. If none is found, the cells fall back to a random policy so the loop still runs. To measure the success rate instead of watching a video, use `scripts/eval_mjlab_policy.py` (see the ML quickstart).

In [ ]:
from pathlib import Path

import numpy as np

from myosuite.utils import gym
from myosuite.utils.checkpoint_utils import find_checkpoint, load_policy
from myosuite.utils.video_io import show_video, write_video

ENV_ID = "myoElbowPose1D6MRandom-v0"
CHECKPOINT = None  # optional: a model_<iter>.pt, an mjlab run directory, or an SB3 .zip
SB3_ZIP = "files/2.1/ElbowPose_policy.zip"  # SB3 checkpoint to try (notebook 2.1 saves the elbow one)
N_EPISODES = 6  # episodes of the rollout

cwd = Path.cwd()
repo_root = cwd if (cwd / "myosuite").is_dir() else cwd.parent

In [ ]:
roots = (cwd, repo_root, repo_root / "tutorials")
env = gym.make(ENV_ID, render_mode="rgb_array")
env.reset()
act = load_policy(env, find_checkpoint(ENV_ID, CHECKPOINT, roots, SB3_ZIP))

In [ ]:
frames = []
for ep in range(N_EPISODES):
    obs, _ = env.reset(seed=ep)  # each reset samples the env's own new target
    done = False
    while not done:
        frames.append(env.render())
        obs, reward, terminated, truncated, info = env.step(act(obs))
        done = terminated or truncated
    print(f"Episode {ep}: solved={info.get('solved')}")
env.close()

Path("videos").mkdir(exist_ok=True)
write_video("videos/trained_policy.mp4", np.asarray(frames), outputdict={"-pix_fmt": "yuv420p"})

In [ ]:
show_video('videos/trained_policy.mp4')